tarea 5

In [19]:
# nombre: sofía rojas
# rut: 22.532.158-2

In [20]:
# importamos los paquetes que utilizaremos

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [21]:
from pathlib import Path
import csv
import json
import pickle
import importlib.util

In [109]:
# creamos la clase 

class CatalogoEstelar:
    def __init__(self, ruta_csv):
        self.ruta_csv = ruta_csv
        datos = np.loadtxt(ruta_csv, dtype= str, delimiter=",", skiprows=1, encoding="utf-8") # lectura del archivo csv, saltamos la primera fila ya que ahi está el encabezado

        self.id = datos[:, 0]
        self.nombre = datos[:, 1]
        self.temperatura_K = datos[:, 2].astype(float) # transformamos los datos
        self.luminosidad_Lsol = datos[:, 3].astype(float) # a float
        self.distancia_pc = datos[:, 4].astype(float)

        self.limpiar_datos() # llamamos al método de limpieza para el análisis que haremos más adelante

    # # # # # # # # # # # # # # # # # # # # # # 
    def limpiar_datos(self):
        #a) para valores no finitos
        nf = (np.isfinite(self.temperatura_K)
        & np.isfinite(self.luminosidad_Lsol)
        & np.isfinite(self.distancia_pc))

        #b) para valores no positivos
        no_p = ((self.temperatura_K > 0)
         & (self.luminosidad_Lsol > 0)
         & (self.distancia_pc > 0))
        
        #c) para duplicados
        mascara = np.zeros_like(self.id, dtype=bool)
        mascara[np.unique(self.id, return_index=True)[1]] = True
        d = mascara
        
        datos_validos = nf & no_p & d # máscara que nos dará los datos del .csv válidos bajo las condiciones de limpieza
        self.id = self.id[datos_validos]
        self.nombre = self.nombre[datos_validos]
        self.temperatura_K = self.temperatura_K[datos_validos]
        self.luminosidad_Lsol = self.luminosidad_Lsol[datos_validos]
        self.distancia_pc = self.distancia_pc[datos_validos]

    # # # # # # # # # # # # # # # # # # # # # # 

            # para guardar las columnas más rápido

    def columnas(self):
        return {"id": self.id,
                "nombre": self.nombre,
                "temperatura_K": self.temperatura_K,
                "luminosidad_Lsol": self.luminosidad_Lsol,
                "distancia_pc": self.distancia_pc}   
    
    
    # # # # # # # # # # # # # # # # # # # # # # 

    def resumen(self):

        total = len(self.id) # n° total de estrellas
        T_media = np.mean(self.temperatura_K) # Temperatura media
        T_max = np.max(self.temperatura_K) # Temperatura máxima
        L_med = np.mean(self.luminosidad_Lsol) # luminosidad media
        d_min = np.min(self.distancia_pc) # distancia mínima
        d_max = np.max(self.distancia_pc) # distancia máxima

        # para obtener el nombre de la estrella más caliente tomamos su posicion:
        
        p_tmax = np.argmax(self.temperatura_K)
        p_Dmin = np.argmin(self.distancia_pc)

        # y entonces :

        e_tmax = self.temperatura_K[p_tmax]
        e_Dmin = self.distancia_pc[p_Dmin]

        # resultadoss : 
        print("--- RESUMEN DEL CATÁLOGO ---")
        print(f' N° total de estrellas: {total}')
        print(f' Temperatura media: {T_media} (K) ')
        print(f' Temperatura máxima: {T_max} (K)  en {e_tmax}')
        print(f' Luminosidad media: {L_med}')
        print(f' Distancia mínima: {d_min} (pc) en {e_Dmin}')
        print(f' Distancia máxima: {d_max} (pc)')

        # # # # # # # # # # # # # # # # # # # # # # 

    # Lectura y escritura de archivos

    def crear_verificarcarpeta(self):
        carpeta = Path.cwd() / "resultados"
        carpeta.mkdir(parents=True, exist_ok=True)
        return carpeta 
    
    # Guardar en CSV
        
    def guardar_csv(self):
        carpeta = self.crear_verificarcarpeta() # llamamos a la función que asegura la ruta correcta
        catalogo_csv = carpeta / "catalogo_copia.csv"
        columnas = np.column_stack([self.id,
        self.nombre, self.temperatura_K, self.luminosidad_Lsol, self.distancia_pc]
        )
        np.savetxt(catalogo_csv, columnas, delimiter=",", header="id, nombre,temperatura_K, luminosidad_Lsol, distancia_pc"
        , comments="",fmt="%s")
        return catalogo_csv

    # Guardar en NPY 

    def guardar_npy(self):
        carpeta = self.crear_verificarcarpeta()
        temperaturas_npy = carpeta / "temperaturas.npy"
        np.save(temperaturas_npy, self.temperatura_K) # guardamos las temperaturas
        return temperaturas_npy
    
    def leer_npy(self):
        carpeta = self.crear_verificarcarpeta()
        return np.load(carpeta / "temperaturas.npy") # Lectura del archivo con numpy load

    # Guardar en NPZ

    def guardar_npz(self):
        carpeta = self.crear_verificarcarpeta()
        cat_completo = carpeta / "catalogo_completo.npz"
        np.savez_compressed(cat_completo, **self.columnas()) # utilizamos ** para copiar más rápido las columnas
        return cat_completo

    def leer_npz(self):
        carpeta = self.crear_verificarcarpeta()
        np.load(carpeta / "catalogo_completo.npz") 
        with np.load(carpeta / "catalogo_completo.npz") as c:
            datos = {
                "id": c["id"],
                "nombre": c["nombre"],
                "temperatura_K": c["temperatura_K"],
                "luminosidad_Lsol": c["luminosidad_Lsol"],
                "distancia_pc": c["distancia_pc"]
            }
        return datos
    # Guardar en Pickle
    def guardar_pickle(self):
        carpeta = self.crear_verificarcarpeta()
        catalogo_pkl = carpeta / "catalogo_completo.pkl"
        with open(catalogo_pkl, "wb") as p:
            pickle.dump(self.columnas(), p)
        return catalogo_pkl    
    # # # # # # # # # # # # # # # # # # # # # # 

    # Verificar y comparar formatos
    def verificar_archivos(self):
        carpeta = Path("resultados")

        # 1. CSV:
        copia = CatalogoEstelar(carpeta/"catalogo_copia.csv") # presentación de resultados

        ok_csv = (len(copia.id) == len(self.id) # utilizamos np.array_equal
              and np.array_equal(copia.id, self.id) # y np.allclose
              and np.array_equal(copia.nombre, self.nombre) # para buscar si coinciden o no
              and np.allclose(copia.temperatura_K, self.temperatura_K) # los datos
              and np.allclose(copia.luminosidad_Lsol, self.luminosidad_Lsol)
              and np.allclose(copia.distancia_pc, self.distancia_pc))
        
        print("CSV   :", "coincide" if ok_csv else "NO coincide")
        # 2. NPY:
        temp = np.load(carpeta/"temperaturas.npy")
        ok_npy = len(copia.id) == len(self.id) and np.allclose(temp, self.temperatura_K)
        print("NPY   :", "coincide" if ok_npy else "NO coincide")

        # 3. NPZ:
        with np.load(carpeta / "catalogo_completo.npz") as z:
            ok_npz = (len(z["id"]) == len(self.id)
                    and np.array_equal(z["id"], self.id)
                    and np.array_equal(z["nombre"], self.nombre)
                    and np.allclose(z["temperatura_K"], self.temperatura_K)
                    and np.allclose(z["luminosidad_Lsol"], self.luminosidad_Lsol)
                    and np.allclose(z["distancia_pc"], self.distancia_pc))
            print("NPZ   :", "coincide" if ok_npz else "NO coincide")

        # 4. PICKLE:
        with open(carpeta / "catalogo_completo.pkl", "rb") as p:
            d = pickle.load(p)
            ok_pkl = (len(d["id"]) == len(self.id)
                    and np.array_equal(d["id"], self.id)
                    and np.array_equal(d["nombre"], self.nombre)
                    and np.allclose(d["temperatura_K"], self.temperatura_K)
                    and np.allclose(d["luminosidad_Lsol"], self.luminosidad_Lsol)
                    and np.allclose(d["distancia_pc"], self.distancia_pc))
            print("Pickle:", "coincide" if ok_pkl else "NO coincide")
    
    #5.B COMPARAR FORMATO
    def comparar_formatos(self):
        carpeta = Path("resultados") # traemos nuestra carpeta en la que están los archivos
        s_csv = (carpeta / "catalogo_copia.csv").stat().st_size # y calculamos
        s_npy = (carpeta / "temperaturas.npy").stat().st_size # el peso
        s_npz = (carpeta / "catalogo_completo.npz").stat().st_size # de cada 
        s_pkl = (carpeta / "catalogo_completo.pkl").stat().st_size # uno 

        print(f'CSV: {s_csv} bytes -- NPY: {s_npy} bytes -- NPZ: {s_npz} bytes -- PKL: {s_pkl} bytes')

Uso de archivos generados

In [ ]:
# Probamos nuestro código(recibimos todos los archivos)
catalogo = CatalogoEstelar("catalogo_estelar.csv")
catalogo.guardar_csv()
catalogo.guardar_npy()
catalogo.guardar_npz()
catalogo.guardar_pickle()
# 
catalogo.verificar_archivos()
catalogo.comparar_formatos()

CSV   : coincide
NPY   : coincide
NPZ   : coincide
Pickle: coincide
CSV: 7049 bytes -- NPY: 864 bytes -- NPZ: 3844 bytes -- PKL: 16516 bytes


Preguntas escritas

1. Diferencias entre CSV, NPY, NPZ y Pickle.
- **CSV:** Es uno de lo más legibles tanto a usuarios como a programas, podría ser un incoveniente tener que transformar el texto a número en caso de necesitarse, es pesado comparado con npy o npz.
- **NPY:** Funciona sólo con numpy y trabaja con un arreglo, no guarda nombres o columnas. Es el archivo más liviano entre los cuatro.
- **NPZ:** También trabaja con numpy, pero recibe más arreglos y almacena los nombres de las columnas como claves.
- **PICKLE:* Serializa bastantes objetos de python, se utilizan modos binarios y hay que desencriptarlo para poder leerlo, es el archivo más pesado.

2. Formato para enviar a otros usuarios
- Opino que el formato '.csv' es el más universal para compartir archivos con otras personas ya que no requiere de tanto procedimiento para acceder a él, es agradable a la vista, además de tener no tanto peso considerando su contenido.

3. Tipo de archivo para varios arreglos
- El mejor a mi parecer es .npz ya que al recibir los arreglos los almacena con su nombre y su tipo, se puede comprimir y no es tan pesado.